<a href="https://colab.research.google.com/github/Ashokyadhav/Econocausal/blob/ashok/DoubleML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_csv("/content/wcer (1).csv")

In [2]:


print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())


Dataset shape:
(4751202, 5)

Column names:
['STORE', 'UPC', 'WEEK', 'MOVE', 'PRICE']

First 5 rows:


,STORE,UPC,WEEK,MOVE,PRICE
0,51,317,372,1,26.020
1,74,317,367,1,19.060
2,93,317,364,2,1.475
3,104,317,383,1,3.170
4,111,317,381,1,10.460



Missing values:
STORE    0
UPC      0
WEEK     0
MOVE     0
PRICE    0
dtype: int64


In [3]:

# Step 3: Understand our dataset

print("Dataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe())

print("\nNumber of unique stores:", df["STORE"].nunique())
print("Number of unique products:", df["UPC"].nunique())
print("Number of unique weeks:", df["WEEK"].nunique())


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4751202 entries, 0 to 4751201
Data columns (total 5 columns):
 #   Column  Dtype  
---  ------  -----  
 0   STORE   int64  
 1   UPC     int64  
 2   WEEK    int64  
 3   MOVE    int64  
 4   PRICE   float64
dtypes: float64(1), int64(4)
memory usage: 181.2 MB

Statistical summary:


,STORE,UPC,WEEK,MOVE,PRICE
count,4.751202e+06,4.751202e+06,4.751202e+06,4.751202e+06,4.751202e+06
mean,8.345552e+01,3.756107e+09,1.960909e+02,1.956532e+01,3.114450e+00
std,3.651187e+01,7.449993e+09,1.149442e+02,5.872462e+01,7.643148e-01
min,2.000000e+00,3.170000e+02,1.000000e+00,1.000000e+00,5.000000e-02
25%,5.600000e+01,1.600069e+09,1.000000e+02,7.000000e+00,2.650000e+00
50%,8.900000e+01,3.800001e+09,1.860000e+02,1.300000e+01,3.140000e+00
75%,1.130000e+02,3.800013e+09,3.100000e+02,2.100000e+01,3.590000e+00
max,1.460000e+02,7.916692e+10,3.990000e+02,1.868800e+04,2.602000e+01



Number of unique stores: 93
Number of unique products: 489
Number of unique weeks: 366


In [4]:

# Step 5: Check MOVE distribution

print("MOVE percentiles:")
display(df["MOVE"].quantile([0.50, 0.90, 0.95, 0.99, 0.999]))

print("\nLargest 10 MOVE values:")
display(df.nlargest(10, "MOVE")[["STORE", "UPC", "WEEK", "MOVE", "PRICE"]])


MOVE percentiles:


,MOVE
0.500,13.0
0.900,34.0
0.950,46.0
0.990,135.0
0.999,569.0



Largest 10 MOVE values:


,STORE,UPC,WEEK,MOVE,PRICE
767425,44,1600066610,13,18688,2.68
2265798,44,3800000510,13,17920,1.85
1922880,86,3000006560,395,17824,1.50
2154815,44,3800000110,13,15360,1.49
2727780,44,3800001611,13,15104,2.39
827749,44,1600066750,13,14080,3.57
657555,44,1600066310,13,12800,1.85
2293603,44,3800000520,13,12544,2.20
2214827,44,3800000127,13,12288,2.41
741901,64,1600066590,30,11776,2.85


In [5]:

# Step 6: Check the relationship between PRICE and MOVE

print("PRICE summary:")
print(df["PRICE"].describe())

print("\nCorrelation between PRICE and MOVE:")
print(df[["PRICE", "MOVE"]].corr())


PRICE summary:
count    4.751202e+06
mean     3.114450e+00
std      7.643148e-01
min      5.000000e-02
25%      2.650000e+00
50%      3.140000e+00
75%      3.590000e+00
max      2.602000e+01
Name: PRICE, dtype: float64

Correlation between PRICE and MOVE:
          PRICE      MOVE
PRICE  1.000000 -0.099806
MOVE  -0.099806  1.000000


In [6]:

# Step 8: Average price by week

weekly_price = (
    df.groupby("WEEK")["PRICE"]
    .mean()
    .sort_index()
)

print("First 10 weeks:")
display(weekly_price.head(10))

print("\nLast 10 weeks:")
display(weekly_price.tail(10))

print("\nNumber of weeks with price data:", weekly_price.count())


First 10 weeks:


,PRICE
WEEK,
1,2.605674
2,2.605345
3,2.605234
4,2.621838
5,2.623008
6,2.621803
7,2.630338
8,2.629094
9,2.636263



Last 10 weeks:


,PRICE
WEEK,
390,3.194828
391,3.190778
392,3.188336
393,3.160984
394,3.160984
395,3.152996
396,3.182525
397,3.157225
398,3.174431



Number of weeks with price data: 366


In [7]:

# Step 9: Check price variation within products

product_price = (
    df.groupby("UPC")["PRICE"]
    .agg(["min", "mean", "max", "std"])
)

print("Products with the highest price variation:")
display(
    product_price.sort_values("std", ascending=False).head(10)
)

print("\nProducts with no price variation:")
print((product_price["std"].fillna(0) == 0).sum())


Products with the highest price variation:


,min,mean,max,std
UPC,,,,
317,1.475,12.037000,26.02,10.458409
3800054268,3.720,6.229333,6.49,0.699945
79166919305,1.660,3.093603,3.58,0.693711
1780019541,1.680,2.929835,3.49,0.688500
3800001727,0.090,4.353422,5.05,0.685618
3828125030,1.490,2.626796,3.39,0.682712
1780010605,0.740,3.567400,3.99,0.572462
3800013930,1.340,4.367080,5.40,0.571532
79166910359,0.640,3.662361,3.99,0.560446



Products with no price variation:
42


In [8]:

# Step 10: Check price variation within each store-product pair

store_product_price = (
    df.groupby(["STORE", "UPC"])["PRICE"]
    .agg(["min", "max", "std", "count"])
)

store_product_price["price_range"] = (
    store_product_price["max"] - store_product_price["min"]
)

print("Total store-product pairs:", len(store_product_price))

print("\nPairs with no price variation:")
print((store_product_price["price_range"] == 0).sum())

print("\nPairs with price variation:")
print((store_product_price["price_range"] > 0).sum())

print("\nExamples with the highest price variation:")
display(
    store_product_price.sort_values(
        "price_range", ascending=False
    ).head(10)
)


Total store-product pairs: 36494

Pairs with no price variation:
5292

Pairs with price variation:
31202

Examples with the highest price variation:


min   max       std  count  price_range
STORE UPC                                                 
100   3800004900  1.64  9.71  0.550870    361         8.07
98    3800001727  0.09  4.89  0.675770    162         4.80
      3800002970  0.10  4.79  0.617819    105         4.69
59    3800000127  0.69  4.96  0.464022    362         4.27
128   3800054276  3.29  7.38  0.993243     20         4.09
114   3800054276  2.93  6.98  0.593332     49         4.05
112   3800001510  0.99  5.00  0.371437    366         4.01
115   3800001510  0.99  5.00  0.392843    364         4.01
51    3800001510  0.99  5.00  0.399313    359         4.01
95    3828125069  0.99  5.00  0.513058    219         4.01

In [9]:

# Step 11: Check duplicate store-product-week combinations

duplicate_count = df.duplicated(
    subset=["STORE", "UPC", "WEEK"]
).sum()

print("Duplicate store-product-week rows:", duplicate_count)

print("\nExample duplicate records:")
display(
    df[
        df.duplicated(
            subset=["STORE", "UPC", "WEEK"],
            keep=False
        )
    ].head(10)
)


Duplicate store-product-week rows: 0

Example duplicate records:


,STORE,UPC,WEEK,MOVE,PRICE


In [10]:

# Step 13: Check records per store and product

store_counts = df["STORE"].value_counts()
product_counts = df["UPC"].value_counts()

print("Records per store:")
print(store_counts.describe())

print("\nRecords per product:")
print(product_counts.describe())

print("\nStores with the most records:")
display(store_counts.head(10))

print("\nProducts with the most records:")
display(product_counts.head(10))


Records per store:
count       93.000000
mean     51088.193548
std      14410.693032
min       4521.000000
25%      53059.000000
50%      56580.000000
75%      58350.000000
max      62575.000000
Name: count, dtype: float64

Records per product:
count      489.000000
mean      9716.159509
std      10534.490437
min          1.000000
25%       1093.000000
50%       4771.000000
75%      17080.000000
max      30100.000000
Name: count, dtype: float64

Stores with the most records:


,count
STORE,
126,62575
74,61560
121,61364
132,61323
131,60940
110,60844
98,60843
59,60582
102,60188



Products with the most records:


,count
UPC,
3800001912,30100
3800000120,30077
3800001621,30074
3800001611,30073
1600065700,30036
3800001411,30033
1600065890,30029
1600066210,30027
3800001720,30025


In [11]:

# Step 14: Check observations per store-product pair

pair_counts = df.groupby(["STORE", "UPC"]).size()

print("Store-product pair summary:")
print(pair_counts.describe())

print("\nPairs with fewer than 50 observations:")
print((pair_counts < 50).sum())

print("\nPairs with at least 300 observations:")
print((pair_counts >= 300).sum())

print("\nPairs with the fewest observations:")
display(pair_counts.sort_values().head(10))


Store-product pair summary:
count    36494.000000
mean       130.191319
std        124.304612
min          1.000000
25%         26.000000
50%         78.000000
75%        227.000000
max        366.000000
dtype: float64

Pairs with fewer than 50 observations:
14588

Pairs with at least 300 observations:
6284

Pairs with the fewest observations:


,,0
STORE,UPC,
84,3800032100,1
123,3800054280,1
32,3800001711,1
18,1780010508,1
124,1600062740,1
95,3800054280,1
114,1600062740,1
12,3828125030,1
47,3800004270,1


In [12]:

# Step 15: Understand the MOVE outcome

print("Minimum MOVE:", df["MOVE"].min())
print("Maximum MOVE:", df["MOVE"].max())
print("Number of unique MOVE values:", df["MOVE"].nunique())

print("\nNumber of zero MOVE values:", (df["MOVE"] == 0).sum())

print("\nMOVE percentiles:")
display(df["MOVE"].quantile([0.25, 0.50, 0.75, 0.90, 0.99, 0.999]))


Minimum MOVE: 1
Maximum MOVE: 18688
Number of unique MOVE values: 1653

Number of zero MOVE values: 0

MOVE percentiles:


,MOVE
0.250,7.0
0.500,13.0
0.750,21.0
0.900,34.0
0.990,135.0
0.999,569.0


In [13]:

# Step 16: Check price and movement variation

pair_variation = (
    df.groupby(["STORE", "UPC"])
    .agg(
        price_std=("PRICE", "std"),
        move_std=("MOVE", "std"),
        observations=("WEEK", "count")
    )
)

print("Pairs with price variation:")
print((pair_variation["price_std"] > 0).sum())

print("\nPairs with movement variation:")
print((pair_variation["move_std"] > 0).sum())

print("\nPairs with both price and movement variation:")
print(
    (
        (pair_variation["price_std"] > 0) &
        (pair_variation["move_std"] > 0)
    ).sum()
)


Pairs with price variation:
31202

Pairs with movement variation:
35728

Pairs with both price and movement variation:
31180


In [14]:

# Step 17: Prepare consecutive-week changes

# Keep only the columns we need
panel_df = df[["STORE", "UPC", "WEEK", "MOVE", "PRICE"]].copy()

# Sort each product's records within each store by week
panel_df = panel_df.sort_values(
    ["STORE", "UPC", "WEEK"]
)

# Calculate changes within each store-product pair
grouped = panel_df.groupby(["STORE", "UPC"])

panel_df["week_gap"] = grouped["WEEK"].diff()
panel_df["d_MOVE"] = grouped["MOVE"].diff()
panel_df["d_PRICE"] = grouped["PRICE"].diff()

# Keep only observations from consecutive weeks
dml_df = panel_df.loc[
    panel_df["week_gap"] == 1,
    ["STORE", "UPC", "WEEK", "d_MOVE", "d_PRICE"]
].copy()

print("Original rows:", len(df))
print("Consecutive-week observations:", len(dml_df))

print("\nFirst 5 observations:")
display(dml_df.head())


Original rows: 4751202
Consecutive-week observations: 4569690

First 5 observations:


,STORE,UPC,WEEK,d_MOVE,d_PRICE
6,2,1313000002,2,2.0,0.0
7,2,1313000002,3,-4.0,0.0
8,2,1313000002,4,8.0,0.0
9,2,1313000002,5,2.0,0.0
10,2,1313000002,6,-14.0,0.0


In [15]:

# Step 18: Check price changes

print("Total observations:", len(dml_df))

print(
    "Price unchanged:",
    (dml_df["d_PRICE"] == 0).sum()
)

print(
    "Price increased:",
    (dml_df["d_PRICE"] > 0).sum()
)

print(
    "Price decreased:",
    (dml_df["d_PRICE"] < 0).sum()
)

print(
    "\nUnique price changes:",
    dml_df["d_PRICE"].nunique()
)


Total observations: 4569690
Price unchanged: 3845677
Price increased: 412190
Price decreased: 311823

Unique price changes: 2385


In [16]:

# Step 19: Check changes in price and movement

print("Price change summary:")
display(dml_df["d_PRICE"].describe())

print("\nMovement change summary:")
display(dml_df["d_MOVE"].describe())

print("\nMissing values:")
print(dml_df[["d_PRICE", "d_MOVE"]].isnull().sum())


Price change summary:


,d_PRICE
count,4.569690e+06
mean,-1.573915e-05
std,2.900081e-01
min,-7.700000e+00
25%,0.000000e+00
50%,0.000000e+00
75%,0.000000e+00
max,5.600000e+00



Movement change summary:


,d_MOVE
count,4.569690e+06
mean,-1.266014e-01
std,8.107699e+01
min,-1.861400e+04
25%,-4.000000e+00
50%,0.000000e+00
75%,4.000000e+00
max,1.862300e+04



Missing values:
d_PRICE    0
d_MOVE     0
dtype: int64


In [17]:

# Step 20: Check large changes in movement

print("Changes above +500:")
print((dml_df["d_MOVE"] > 500).sum())

print("\nChanges below -500:")
print((dml_df["d_MOVE"] < -500).sum())

print("\nLargest positive changes:")
display(
    dml_df.nlargest(5, "d_MOVE")
)

print("\nLargest negative changes:")
display(
    dml_df.nsmallest(5, "d_MOVE")
)


Changes above +500:
5281

Changes below -500:
5259

Largest positive changes:


,STORE,UPC,WEEK,d_MOVE,d_PRICE
767425,44,1600066610,13,18623.0,0.00
2265798,44,3800000510,13,17842.0,0.00
1922880,86,3000006560,395,17812.0,-1.85
2154815,44,3800000110,13,15299.0,0.00
2727780,44,3800001611,13,15065.0,0.00



Largest negative changes:


,STORE,UPC,WEEK,d_MOVE,d_PRICE
767426,44,1600066610,14,-18614.0,0.22
2265799,44,3800000510,14,-17844.0,0.00
1922881,86,3000006560,396,-17359.0,1.69
2154816,44,3800000110,14,-15273.0,0.00
2727781,44,3800001611,14,-15045.0,0.00


In [18]:

# Step 21: Compare movement changes by price-change status

dml_df["price_status"] = "Unchanged"

dml_df.loc[
    dml_df["d_PRICE"] > 0, "price_status"
] = "Increased"

dml_df.loc[
    dml_df["d_PRICE"] < 0, "price_status"
] = "Decreased"

summary = dml_df.groupby("price_status")["d_MOVE"].agg(
    ["count", "mean", "median", "std"]
)

display(summary)


,count,mean,median,std
price_status,,,,
Decreased,311823,48.066073,8.0,146.347910
Increased,412190,-33.671477,-4.0,146.471773
Unchanged,3845677,-0.438828,0.0,58.871901


In [19]:

%pip install DoubleML


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.8/607.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.4/270.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.7/531.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.0 MB/s eta 0:00:00


In [20]:

import doubleml as dml

print("DoubleML version:", dml.__version__)
print("DoubleML imported successfully!")


DoubleML version: 0.11.4
DoubleML imported successfully!


In [21]:

# Step 23: Create a sample for initial model development

sample_size = 200_000

dml_sample = dml_df.sample(
    n=min(sample_size, len(dml_df)),
    random_state=42
).copy()

print("Full prepared dataset:", len(dml_df))
print("Sample size:", len(dml_sample))

print("\nSample preview:")
display(dml_sample.head())


Full prepared dataset: 4569690
Sample size: 200000

Sample preview:


,STORE,UPC,WEEK,d_MOVE,d_PRICE,price_status
64995,133,1313000050,157,1.0,0.62,Increased
1678838,122,3000002550,233,6.0,0.00,Unchanged
3840009,132,3828125029,133,3.0,-0.04,Decreased
3073266,12,3800003700,156,6.0,-0.85,Decreased
3210046,119,3800004510,329,5.0,0.00,Unchanged


In [22]:

# Step 24: Prepare outcome, treatment, and controls

Y = dml_sample["d_MOVE"].astype(float)

T = dml_sample["d_PRICE"].astype(float)

X = dml_sample[["STORE", "UPC", "WEEK"]].copy()

# Treat identifiers as categories
for col in ["STORE", "UPC", "WEEK"]:
    X[col] = X[col].astype("category")

print("Outcome shape:", Y.shape)
print("Treatment shape:", T.shape)
print("Control variables shape:", X.shape)

print("\nControl variable data types:")
print(X.dtypes)


Outcome shape: (200000,)
Treatment shape: (200000,)
Control variables shape: (200000, 3)

Control variable data types:
STORE    category
UPC      category
WEEK     category
dtype: object


In [23]:

# Step 25: One-hot encode the control variables

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_encoded = encoder.fit_transform(X)

print("Original control shape:", X.shape)
print("Encoded control shape:", X_encoded.shape)
print("Sparse matrix:", X_encoded.__class__.__name__)


Original control shape: (200000, 3)
Encoded control shape: (200000, 909)
Sparse matrix: csr_matrix


In [24]:

# Step 26: Check installed DML classes

import doubleml as dml

print("DoubleML version:", dml.__version__)

print("\nAvailable DML classes:")
print([
    name for name in dir(dml)
    if name.startswith("DoubleML")
])


DoubleML version: 0.11.4

Available DML classes:
['DoubleMLAPO', 'DoubleMLAPOS', 'DoubleMLBLP', 'DoubleMLCVAR', 'DoubleMLClusterData', 'DoubleMLCore', 'DoubleMLDID', 'DoubleMLDIDCS', 'DoubleMLDIDData', 'DoubleMLData', 'DoubleMLFramework', 'DoubleMLIIVM', 'DoubleMLIRM', 'DoubleMLLPLR', 'DoubleMLLPQ', 'DoubleMLPLIV', 'DoubleMLPLPR', 'DoubleMLPLR', 'DoubleMLPQ', 'DoubleMLPanelData', 'DoubleMLPolicyTree', 'DoubleMLQTE', 'DoubleMLRDDData', 'DoubleMLSSM', 'DoubleMLSSMData']


In [25]:

# Step 27: Inspect cluster-aware DML data support

import inspect
import doubleml as dml

print("Cluster data class:")
print(dml.DoubleMLClusterData)

print("\nConstructor signature:")
print(inspect.signature(dml.DoubleMLClusterData))


Cluster data class:
<class 'doubleml.data.DoubleMLClusterData'>

Constructor signature:
(data, y_col, d_cols, cluster_cols, x_cols=None, z_cols=None, t_col=None, s_col=None, use_other_treat_as_covariate=True, force_all_x_finite=True)


In [26]:

# Step 28: Prepare data for cluster-aware DML

cluster_df = dml_sample[
    ["STORE", "UPC", "WEEK", "d_MOVE", "d_PRICE"]
].copy()

# Create a unique identifier for each store-product pair
cluster_df["PAIR_ID"] = (
    cluster_df["STORE"].astype(str)
    + "_"
    + cluster_df["UPC"].astype(str)
)

print("Rows:", len(cluster_df))
print("Unique store-product pairs:", cluster_df["PAIR_ID"].nunique())

print("\nFirst 5 rows:")
display(cluster_df.head())


Rows: 200000
Unique store-product pairs: 28394

First 5 rows:


,STORE,UPC,WEEK,d_MOVE,d_PRICE,PAIR_ID
64995,133,1313000050,157,1.0,0.62,133_1313000050
1678838,122,3000002550,233,6.0,0.00,122_3000002550
3840009,132,3828125029,133,3.0,-0.04,132_3828125029
3073266,12,3800003700,156,6.0,-0.85,12_3800003700
3210046,119,3800004510,329,5.0,0.00,119_3800004510


In [27]:

# Step 29: Read the cluster-data documentation

help(dml.DoubleMLClusterData)


Help on class DoubleMLClusterData in module doubleml.data:

class DoubleMLClusterData(doubleml.data.base_data.DoubleMLData)
 |  DoubleMLClusterData(
 |      data,
 |      y_col,
 |      d_cols,
 |      cluster_cols,
 |      x_cols=None,
 |      z_cols=None,
 |      t_col=None,
 |      s_col=None,
 |      use_other_treat_as_covariate=True,
 |      force_all_x_finite=True
 |  )
 |
 |  Backwards compatibility wrapper for DoubleMLData with cluster_cols.
 |  This class is deprecated and will be removed in a future version.
 |  Use DoubleMLData with cluster_cols instead.
 |
 |  Method resolution order:
 |      DoubleMLClusterData
 |      doubleml.data.base_data.DoubleMLData
 |      doubleml.data.base_data.DoubleMLBaseData
 |      abc.ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      data,
 |      y_col,
 |      d_cols,
 |      cluster_cols,
 |      x_cols=None,
 |      z_cols=None,
 |      t_col=None,
 |      s_col=None,
 |      use_other_treat_a

In [28]:

# Step 30: Inspect the current DoubleMLData API

import inspect
import doubleml as dml

print("DoubleMLData constructor:")
print(inspect.signature(dml.DoubleMLData))


DoubleMLData constructor:
(data, y_col, d_cols, x_cols=None, z_cols=None, cluster_cols=None, use_other_treat_as_covariate=True, force_all_x_finite=True, force_all_d_finite=True)


In [29]:

# Step 31: Create the DoubleML data object

dml_data = dml.DoubleMLData(
    data=cluster_df,
    y_col="d_MOVE",
    d_cols="d_PRICE",
    x_cols=["STORE", "UPC", "WEEK"],
    cluster_cols=["PAIR_ID"]
)

print("DoubleML data object created successfully!")
print("Outcome:", dml_data.y_col)
print("Treatment:", dml_data.d_cols)
print("Controls:", dml_data.x_cols)
print("Clusters:", dml_data.cluster_cols)


DoubleML data object created successfully!
Outcome: d_MOVE
Treatment: ['d_PRICE']
Controls: ['STORE', 'UPC', 'WEEK']
Clusters: ['PAIR_ID']


In [30]:

# Step 32: Inspect the partially linear regression estimator

import inspect
import doubleml as dml

print("DoubleMLPLR class:")
print(dml.DoubleMLPLR)

print("\nConstructor signature:")
print(inspect.signature(dml.DoubleMLPLR))


DoubleMLPLR class:
<class 'doubleml.plm.plr.DoubleMLPLR'>

Constructor signature:
(obj_dml_data, ml_l, ml_m, ml_g=None, n_folds=5, n_rep=1, score='partialling out', draw_sample_splitting=True)


In [31]:

# Step 33: Inspect DoubleMLPLR options

help(dml.DoubleMLPLR)


Help on class DoubleMLPLR in module doubleml.plm.plr:

class DoubleMLPLR(doubleml.double_ml_score_mixins.LinearScoreMixin, doubleml.double_ml.DoubleML)
 |  DoubleMLPLR(
 |      obj_dml_data,
 |      ml_l,
 |      ml_m,
 |      ml_g=None,
 |      n_folds=5,
 |      n_rep=1,
 |      score='partialling out',
 |      draw_sample_splitting=True
 |  )
 |
 |  Double machine learning for partially linear regression models
 |
 |  Parameters
 |  ----------
 |  obj_dml_data : :class:`DoubleMLData` object
 |      The :class:`DoubleMLData` object providing the data and specifying the variables for the causal model.
 |
 |  ml_l : estimator implementing ``fit()`` and ``predict()``
 |      A machine learner implementing ``fit()`` and ``predict()`` methods (e.g.
 |      :py:class:`sklearn.ensemble.RandomForestRegressor`) for the nuisance function :math:`\ell_0(X) = E[Y|X]`.
 |
 |  ml_m : estimator implementing ``fit()`` and ``predict()``
 |      A machine learner implementing ``fit()`` and ``predict()`

In [32]:

import inspect

print(inspect.signature(dml.DoubleMLPLR.draw_sample_splitting))
print(inspect.signature(dml.DoubleMLPLR.set_sample_splitting))


(self)
(self, all_smpls, all_smpls_cluster=None)


In [33]:

# Step 34: Inspect the sample-splitting implementation

print(inspect.getsource(dml.DoubleMLPLR.set_sample_splitting))


    def set_sample_splitting(self, all_smpls, all_smpls_cluster=None):
        """
        Set the sample splitting for DoubleML models.

        The  attributes ``n_folds`` and ``n_rep`` are derived from the provided partition.

        Parameters
        ----------
        all_smpls : list or tuple
            If nested list of lists of tuples:
                The outer list needs to provide an entry per repeated sample splitting (length of list is set as
                ``n_rep``).
                The inner list needs to provide a tuple (train_ind, test_ind) per fold (length of list is set as
                ``n_folds``). test_ind must form a partition for each inner list.
            If list of tuples:
                The list needs to provide a tuple (train_ind, test_ind) per fold (length of list is set as
                ``n_folds``). test_ind must form a partition. ``n_rep=1`` is always set.
            If tuple:
                Must be a tuple with two elements train_ind and te

In [34]:

# Step 35: Check the cluster structure

print("Unique stores:", cluster_df["STORE"].nunique())
print("Unique products:", cluster_df["UPC"].nunique())
print("Unique store-product pairs:", cluster_df["PAIR_ID"].nunique())
print("Unique weeks:", cluster_df["WEEK"].nunique())

print("\nObservations per store:")
print(cluster_df.groupby("STORE").size().describe())

print("\nObservations per store-product pair:")
print(cluster_df.groupby("PAIR_ID").size().describe())


Unique stores: 93
Unique products: 455
Unique store-product pairs: 28394
Unique weeks: 361

Observations per store:
count      93.000000
mean     2150.537634
std       610.872871
min       195.000000
25%      2208.000000
50%      2349.000000
75%      2472.000000
max      2718.000000
dtype: float64

Observations per store-product pair:
count    28394.000000
mean         7.043742
std          5.676791
min          1.000000
25%          2.000000
50%          5.000000
75%         11.000000
max         32.000000
dtype: float64


In [35]:
print("DML data object:", dml_data)
print("Sample shape:", dml_sample.shape)
print("Cluster data shape:", cluster_df.shape)

DML data object: ================== DoubleMLData Object ==================

------------------ Data summary      ------------------
Outcome variable: d_MOVE
Treatment variable(s): ['d_PRICE']
Covariates: ['STORE', 'UPC', 'WEEK']
Instrument variable(s): None
Cluster variable(s): ['PAIR_ID']
Is cluster data: True
No. Observations: 200000
------------------ DataFrame info    ------------------
<class 'pandas.core.frame.DataFrame'>
Index: 200000 entries, 64995 to 1905521
Columns: 6 entries, STORE to PAIR_ID
dtypes: float64(2), int64(3), object(1)
memory usage: 10.7+ MB

Sample shape: (200000, 6)
Cluster data shape: (200000, 6)


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

print("All required machine learning tools are ready!")

All required machine learning tools are ready!


In [37]:
# Select the categorical control columns
categorical_features = ["STORE", "UPC", "WEEK"]

# Create the preprocessing step
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [38]:
# Model 1: Predict changes in product movement
ml_l = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

# Model 2: Predict changes in price
ml_m = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

print("Both DML model pipelines are ready!")

Both DML model pipelines are ready!


In [39]:
import inspect
import doubleml as dml

print("DoubleML version:", dml.__version__)
print("\nDoubleMLPLR parameters:")
print(inspect.signature(dml.DoubleMLPLR))

DoubleML version: 0.11.4

DoubleMLPLR parameters:
(obj_dml_data, ml_l, ml_m, ml_g=None, n_folds=5, n_rep=1, score='partialling out', draw_sample_splitting=True)


In [40]:
import inspect

print(inspect.signature(dml.DoubleMLPLR.set_sample_splitting))
print("\nCluster-aware splitting documentation:")
print(dml.DoubleMLPLR.set_sample_splitting.__doc__[:1800])

(self, all_smpls, all_smpls_cluster=None)

Cluster-aware splitting documentation:

Set the sample splitting for DoubleML models.

The  attributes ``n_folds`` and ``n_rep`` are derived from the provided partition.

Parameters
----------
all_smpls : list or tuple
    If nested list of lists of tuples:
        The outer list needs to provide an entry per repeated sample splitting (length of list is set as
        ``n_rep``).
        The inner list needs to provide a tuple (train_ind, test_ind) per fold (length of list is set as
        ``n_folds``). test_ind must form a partition for each inner list.
    If list of tuples:
        The list needs to provide a tuple (train_ind, test_ind) per fold (length of list is set as
        ``n_folds``). test_ind must form a partition. ``n_rep=1`` is always set.
    If tuple:
        Must be a tuple with two elements train_ind and test_ind. Only viable option is to set
        train_ind and test_ind to np.arange(n_obs), which corresponds to no sample 

In [41]:
from sklearn.model_selection import GroupKFold
import numpy as np

# Create five folds, keeping each store-product pair together
group_kfold = GroupKFold(n_splits=5)

groups = cluster_df["PAIR_ID"].to_numpy()
indices = np.arange(len(cluster_df))

folds = list(
    group_kfold.split(indices, groups=groups)
)

# Check the folds
for i, (train_idx, test_idx) in enumerate(folds, start=1):
    train_groups = set(groups[train_idx])
    test_groups = set(groups[test_idx])

    overlap = train_groups.intersection(test_groups)

    print(
        f"Fold {i}: Train={len(train_idx)}, "
        f"Test={len(test_idx)}, "
        f"Overlapping pairs={len(overlap)}"
    )

Fold 1: Train=160000, Test=40000, Overlapping pairs=0
Fold 2: Train=160000, Test=40000, Overlapping pairs=0
Fold 3: Train=160000, Test=40000, Overlapping pairs=0
Fold 4: Train=160000, Test=40000, Overlapping pairs=0
Fold 5: Train=160000, Test=40000, Overlapping pairs=0


In [42]:
print("Cluster columns:", dml_data.cluster_cols)
print("Unique pair IDs:", cluster_df["PAIR_ID"].nunique())

print("\nCluster variable data type:")
print(cluster_df["PAIR_ID"].dtype)

print("\nFirst 5 pair IDs:")
print(cluster_df["PAIR_ID"].head().tolist())

Cluster columns: ['PAIR_ID']
Unique pair IDs: 28394

Cluster variable data type:
object

First 5 pair IDs:
['133_1313000050', '122_3000002550', '132_3828125029', '12_3800003700', '119_3800004510']


In [43]:
print(dml.DoubleMLPLR.set_sample_splitting.__doc__[1800:3600])

e_plr_CCDDHNR2018(n_obs=10, alpha=0.5)
>>> dml_plr_obj = dml.DoubleMLPLR(obj_dml_data, ml_g, ml_m)
>>> # sample splitting with two folds and cross-fitting
>>> smpls = [([0, 1, 2, 3, 4], [5, 6, 7, 8, 9]),
...          ([5, 6, 7, 8, 9], [0, 1, 2, 3, 4])]
>>> dml_plr_obj.set_sample_splitting(smpls) # doctest: +ELLIPSIS
<doubleml.plm.plr.DoubleMLPLR object at 0x...>
>>> # sample splitting with two folds and repeated cross-fitting with n_rep = 2
>>> smpls = [[([0, 1, 2, 3, 4], [5, 6, 7, 8, 9]),
...           ([5, 6, 7, 8, 9], [0, 1, 2, 3, 4])],
...          [([0, 2, 4, 6, 8], [1, 3, 5, 7, 9]),
...           ([1, 3, 5, 7, 9], [0, 2, 4, 6, 8])]]
>>> dml_plr_obj.set_sample_splitting(smpls) # doctest: +ELLIPSIS
<doubleml.plm.plr.DoubleMLPLR object at 0x...>



In [44]:
doc = dml.DoubleMLPLR.set_sample_splitting.__doc__

print(doc[3600:5400])

In [45]:
doc = dml.DoubleMLPLR.set_sample_splitting.__doc__

print("Documentation length:", len(doc))
print(doc[1200:])

Documentation length: 2559
 nesting contains a tuple of training and
    testing lists. Both training and testing contain an array for each cluster variable, which form a partition of
    the clusters.
    Default is ``None``.

Returns
-------
self : object

Examples
--------
>>> import numpy as np
>>> import doubleml as dml
>>> from doubleml.plm.datasets import make_plr_CCDDHNR2018
>>> from sklearn.ensemble import RandomForestRegressor
>>> from sklearn.base import clone
>>> np.random.seed(3141)
>>> learner = RandomForestRegressor(max_depth=2, n_estimators=10)
>>> ml_g = learner
>>> ml_m = learner
>>> obj_dml_data = make_plr_CCDDHNR2018(n_obs=10, alpha=0.5)
>>> dml_plr_obj = dml.DoubleMLPLR(obj_dml_data, ml_g, ml_m)
>>> # sample splitting with two folds and cross-fitting
>>> smpls = [([0, 1, 2, 3, 4], [5, 6, 7, 8, 9]),
...          ([5, 6, 7, 8, 9], [0, 1, 2, 3, 4])]
>>> dml_plr_obj.set_sample_splitting(smpls) # doctest: +ELLIPSIS
<doubleml.plm.plr.DoubleMLPLR object at 0x...>
>>> # sa

In [46]:
# Get the unique store-product pairs in a fixed order
unique_pairs = np.unique(groups)

print("Total unique pairs:", len(unique_pairs))

# Check how many pairs are in each fold
for i, (_, test_idx) in enumerate(folds, start=1):
    test_pairs = np.unique(groups[test_idx])

    print(
        f"Fold {i}: {len(test_pairs)} test pairs"
    )

Total unique pairs: 28394
Fold 1: 5678 test pairs
Fold 2: 5679 test pairs
Fold 3: 5679 test pairs
Fold 4: 5679 test pairs
Fold 5: 5679 test pairs


In [47]:
print("Cluster data shape:", dml_data.data.shape)
print("Cluster columns:", dml_data.cluster_cols)

print("\nFirst 5 cluster IDs in the data:")
print(dml_data.data["PAIR_ID"].head().tolist())

print("\nUnique cluster count:")
print(dml_data.data["PAIR_ID"].nunique())

Cluster data shape: (200000, 6)
Cluster columns: ['PAIR_ID']

First 5 cluster IDs in the data:
['133_1313000050', '122_3000002550', '132_3828125029', '12_3800003700', '119_3800004510']

Unique cluster count:
28394


In [48]:
print("Cluster-aware data class:")
print(type(dml_data).__name__)

print("\nCluster-related attributes:")
print([
    name for name in dir(dml_data)
    if "cluster" in name.lower()
])

Cluster-aware data class:
DoubleMLData

Cluster-related attributes:
['_check_disjoint_sets_cluster_cols', '_cluster_cols', '_cluster_vars', '_is_cluster_data', '_set_cluster_vars', 'cluster_cols', 'cluster_vars', 'is_cluster_data', 'n_cluster_vars']


In [49]:
print("Cluster variables:", dml_data.cluster_vars)
print("Number of clusters:", dml_data.n_cluster_vars)
print("Cluster columns:", dml_data.cluster_cols)

Cluster variables: [['133_1313000050']
 ['122_3000002550']
 ['132_3828125029']
 ...
 ['136_4300011760']
 ['114_3800000828']
 ['129_3000006500']]
Number of clusters: 1
Cluster columns: ['PAIR_ID']


In [50]:
import inspect

print(inspect.getsource(dml.DoubleMLPLR.set_sample_splitting))

    def set_sample_splitting(self, all_smpls, all_smpls_cluster=None):
        """
        Set the sample splitting for DoubleML models.

        The  attributes ``n_folds`` and ``n_rep`` are derived from the provided partition.

        Parameters
        ----------
        all_smpls : list or tuple
            If nested list of lists of tuples:
                The outer list needs to provide an entry per repeated sample splitting (length of list is set as
                ``n_rep``).
                The inner list needs to provide a tuple (train_ind, test_ind) per fold (length of list is set as
                ``n_folds``). test_ind must form a partition for each inner list.
            If list of tuples:
                The list needs to provide a tuple (train_ind, test_ind) per fold (length of list is set as
                ``n_folds``). test_ind must form a partition. ``n_rep=1`` is always set.
            If tuple:
                Must be a tuple with two elements train_ind and te

In [51]:
import inspect

source = inspect.getsource(
    dml.DoubleMLPLR.set_sample_splitting
)

print(source[source.find("def set_sample_splitting"):])

def set_sample_splitting(self, all_smpls, all_smpls_cluster=None):
        """
        Set the sample splitting for DoubleML models.

        The  attributes ``n_folds`` and ``n_rep`` are derived from the provided partition.

        Parameters
        ----------
        all_smpls : list or tuple
            If nested list of lists of tuples:
                The outer list needs to provide an entry per repeated sample splitting (length of list is set as
                ``n_rep``).
                The inner list needs to provide a tuple (train_ind, test_ind) per fold (length of list is set as
                ``n_folds``). test_ind must form a partition for each inner list.
            If list of tuples:
                The list needs to provide a tuple (train_ind, test_ind) per fold (length of list is set as
                ``n_folds``). test_ind must form a partition. ``n_rep=1`` is always set.
            If tuple:
                Must be a tuple with two elements train_ind and test_i

In [52]:
import inspect
from doubleml.utils._checks import _check_sample_splitting

print(inspect.getsource(_check_sample_splitting))


def _check_sample_splitting(all_smpls, all_smpls_cluster, dml_data, is_cluster_data, n_obs=None):
    # default value for n_obs is None (different for e.g. DoubleMLPanelData)
    if n_obs is None:
        n_obs = dml_data.n_obs
    if isinstance(all_smpls, tuple):
        if not len(all_smpls) == 2:
            raise ValueError(
                "Invalid partition provided. Tuple for train_ind and test_ind must consist of exactly two elements."
            )
        all_smpls = _check_smpl_split_tpl(all_smpls, n_obs)
        if _check_is_partition([all_smpls], n_obs) & _check_is_partition([(all_smpls[1], all_smpls[0])], n_obs):
            n_rep = 1
            n_folds = 1
            smpls = [[all_smpls]]
        else:
            raise ValueError("Invalid partition provided. Tuple provided that doesn't form a partition.")
    else:
        if not isinstance(all_smpls, list):
            raise TypeError(
                f"all_smpls must be of list or tuple type. {str(all_smpls)} of typ

In [53]:
import inspect
from doubleml.utils._checks import _check_sample_splitting

source = inspect.getsource(_check_sample_splitting)

for i, line in enumerate(source.splitlines()):
    if "all_smpls_cluster" in line or "cluster" in line.lower():
        print(f"{i}: {line}")

0: def _check_sample_splitting(all_smpls, all_smpls_cluster, dml_data, is_cluster_data, n_obs=None):
67:     if is_cluster_data:
68:         smpls_cluster = _check_cluster_sample_splitting(all_smpls_cluster, dml_data, n_rep, n_folds)
70:         smpls_cluster = None
72:     return smpls, smpls_cluster, n_rep, n_folds


In [55]:
import doubleml.utils._checks as checks

print([
    name for name in dir(checks)
    if "cluster" in name.lower() and "split" in name.lower()
])

['_check_cluster_sample_splitting']


In [56]:
import inspect
from doubleml.utils._checks import _check_cluster_sample_splitting

print(inspect.getsource(_check_cluster_sample_splitting))

def _check_cluster_sample_splitting(all_smpls_cluster, dml_data, n_rep, n_folds):
    if all_smpls_cluster is None:
        raise ValueError("For cluster data, all_smpls_cluster must be provided.")

    n_rep_cluster = len(all_smpls_cluster)
    if n_rep_cluster != n_rep:
        raise ValueError(
            "Invalid samples provided. Number of repetitions for all_smpls and all_smpls_cluster must be the same."
        )

    for i_rep in range(n_rep):
        n_folds_cluster = len(all_smpls_cluster[i_rep])
        if n_folds_cluster != n_folds:
            raise ValueError("Invalid samples provided. Number of folds for all_smpls and all_smpls_cluster must be the same.")
        for i_cluster in range(dml_data.n_cluster_vars):
            this_cluster_var = dml_data.cluster_vars[:, i_cluster]
            clusters = np.unique(this_cluster_var)
            cluster_partition = [all_smpls_cluster[0][0][0][i_cluster], all_smpls_cluster[0][0][1][i_cluster]]
            is_cluster_partition =

In [57]:
# Get the cluster IDs in DoubleML's own order
cluster_ids = dml_data.cluster_vars[:, 0]

# Map each cluster ID to its position
cluster_to_index = {
    cluster_id: i
    for i, cluster_id in enumerate(cluster_ids)
}

all_smpls_cluster = []

for train_idx, test_idx in folds:
    train_pairs = np.unique(groups[train_idx])
    test_pairs = np.unique(groups[test_idx])

    train_cluster_idx = np.array([
        cluster_to_index[pair] for pair in train_pairs
    ])

    test_cluster_idx = np.array([
        cluster_to_index[pair] for pair in test_pairs
    ])

    all_smpls_cluster.append([
        (train_cluster_idx, test_cluster_idx)
    ])

print("Cluster folds prepared:", len(all_smpls_cluster))
print("First fold — training clusters:",
      len(all_smpls_cluster[0][0][0]))
print("First fold — testing clusters:",
      len(all_smpls_cluster[0][0][1]))

Cluster folds prepared: 5
First fold — training clusters: 22716
First fold — testing clusters: 5678


In [59]:
# Add the repetition level expected by DoubleML
all_smpls = [folds]
all_smpls_cluster_fixed = [all_smpls_cluster]

print("Row-level repetitions:", len(all_smpls))
print("Cluster-level repetitions:", len(all_smpls_cluster_fixed))
print("Folds per repetition:", len(all_smpls[0]))
print("Cluster folds per repetition:", len(all_smpls_cluster_fixed[0]))

Row-level repetitions: 1
Cluster-level repetitions: 1
Folds per repetition: 5
Cluster folds per repetition: 5


In [63]:
print("Number of repetitions:", len(all_smpls_cluster_fixed))
print("Folds in first repetition:", len(all_smpls_cluster_fixed[0]))

print("\nFirst fold structure:")
print(all_smpls_cluster_fixed[0][0])

print("\nLengths of the cluster arrays:")
print([
    len(arr)
    for arr in all_smpls_cluster_fixed[0][0][0]
])

Number of repetitions: 1
Folds in first repetition: 5

First fold structure:
([array([     0,      1,      2, ..., 199997, 199998, 199999]), array([     6,      9,     12, ..., 199976, 199980, 199982])],)

Lengths of the cluster arrays:
[160000, 40000]


In [64]:
# Map each row's pair ID to its cluster index in DoubleML
cluster_ids = dml_data.cluster_vars[:, 0]

cluster_to_index = {
    pair_id: i
    for i, pair_id in enumerate(cluster_ids)
}

# Check that the row IDs can be mapped correctly
row_cluster_indices = np.array([
    cluster_to_index[pair_id]
    for pair_id in groups
])

print("Total rows:", len(row_cluster_indices))
print("Unique cluster indices:", len(np.unique(row_cluster_indices)))
print("Expected unique clusters:", dml_data.n_cluster_vars)
print("First 10 mapped indices:", row_cluster_indices[:10])

Total rows: 200000
Unique cluster indices: 28394
Expected unique clusters: 1
First 10 mapped indices: [186571 163404 194377 194530 195109 190475 188561 154079 182575 199563]


In [65]:
all_smpls_cluster_fixed = [[
    (
        np.unique(row_cluster_indices[train_idx]),
        np.unique(row_cluster_indices[test_idx])
    )
    for train_idx, test_idx in folds
]]

print("Repetitions:", len(all_smpls_cluster_fixed))
print("Folds:", len(all_smpls_cluster_fixed[0]))
print("Training clusters in fold 1:",
      len(all_smpls_cluster_fixed[0][0][0]))
print("Testing clusters in fold 1:",
      len(all_smpls_cluster_fixed[0][0][1]))

Repetitions: 1
Folds: 5
Training clusters in fold 1: 22716
Testing clusters in fold 1: 5678


In [67]:
import inspect
from doubleml.utils._checks import _check_cluster_partitions

print(inspect.getsource(_check_cluster_partitions))

def _check_cluster_partitions(smpls, values):
    test_indices = np.concatenate([test_index for test_index in smpls])
    if len(test_indices) != len(values):
        return False
    if np.any(np.sort(test_indices) != np.sort(values)):
        return False
    return True



In [68]:
# DoubleML expects one pair of lists per fold:
# (training cluster indices, testing cluster indices)

all_smpls_cluster_fixed = [[
    (
        [
            np.unique(row_cluster_indices[train_idx])
        ],
        [
            np.unique(row_cluster_indices[test_idx])
        ]
    )
    for train_idx, test_idx in folds
]]

print("Rebuilt cluster folds:", len(all_smpls_cluster_fixed[0]))
print("Training clusters in fold 1:",
      len(all_smpls_cluster_fixed[0][0][0][0]))
print("Testing clusters in fold 1:",
      len(all_smpls_cluster_fixed[0][0][1][0]))

Rebuilt cluster folds: 5
Training clusters in fold 1: 22716
Testing clusters in fold 1: 5678


In [69]:
first_fold = all_smpls_cluster_fixed[0][0]

print("Fold type:", type(first_fold))
print("Training entry type:", type(first_fold[0]))
print("Testing entry type:", type(first_fold[1]))

print("Training array shape:", first_fold[0][0].shape)
print("Testing array shape:", first_fold[1][0].shape)

print("Training index range:",
      first_fold[0][0].min(), first_fold[0][0].max())
print("Testing index range:",
      first_fold[1][0].min(), first_fold[1][0].max())

Fold type: <class 'tuple'>
Training entry type: <class 'list'>
Testing entry type: <class 'list'>
Training array shape: (22716,)
Testing array shape: (5678,)
Training index range: 18 199999
Testing index range: 385 199982


In [70]:
print("Cluster vars shape:", dml_data.cluster_vars.shape)
print("First 10 cluster vars:")
print(dml_data.cluster_vars[:10, 0])

print("\nUnique cluster IDs in cluster_vars:",
      len(np.unique(dml_data.cluster_vars[:, 0])))

Cluster vars shape: (200000, 1)
First 10 cluster vars:
['133_1313000050' '122_3000002550' '132_3828125029' '12_3800003700'
 '119_3800004510' '68_3800001532' '105_3800001111' '133_3800054280'
 '75_1600066590' '101_3800001411']

Unique cluster IDs in cluster_vars: 28394


In [71]:
# Create a stable list of unique cluster IDs
unique_cluster_ids = np.unique(dml_data.cluster_vars[:, 0])

# Map each cluster ID to its index
cluster_to_index = {
    cluster_id: i
    for i, cluster_id in enumerate(unique_cluster_ids)
}

# Map each row to its cluster index
row_cluster_indices = np.array([
    cluster_to_index[pair_id]
    for pair_id in dml_data.cluster_vars[:, 0]
])

print("Unique clusters:", len(unique_cluster_ids))
print("Mapped cluster IDs:", len(np.unique(row_cluster_indices)))
print("Minimum cluster index:", row_cluster_indices.min())
print("Maximum cluster index:", row_cluster_indices.max())

Unique clusters: 28394
Mapped cluster IDs: 28394
Minimum cluster index: 0
Maximum cluster index: 28393


In [72]:
all_smpls_cluster_fixed = [[
    (
        [
            np.unique(row_cluster_indices[train_idx]).astype(int)
        ],
        [
            np.unique(row_cluster_indices[test_idx]).astype(int)
        ]
    )
    for train_idx, test_idx in folds
]]

print("Folds prepared:", len(all_smpls_cluster_fixed[0]))
print("Training clusters in fold 1:",
      len(all_smpls_cluster_fixed[0][0][0][0]))
print("Testing clusters in fold 1:",
      len(all_smpls_cluster_fixed[0][0][1][0]))

Folds prepared: 5
Training clusters in fold 1: 22716
Testing clusters in fold 1: 5678


In [73]:
# Check whether test folds cover all clusters exactly once
test_cluster_indices = np.concatenate([
    all_smpls_cluster_fixed[0][i][1][0]
    for i in range(5)
])

print("Total test-cluster indices:", len(test_cluster_indices))
print("Unique test clusters:", len(np.unique(test_cluster_indices)))
print("Expected unique clusters:", len(unique_cluster_ids))
print("Every cluster appears once:",
      len(test_cluster_indices) == len(np.unique(test_cluster_indices))
      and len(np.unique(test_cluster_indices)) == len(unique_cluster_ids))

Total test-cluster indices: 28394
Unique test clusters: 28394
Expected unique clusters: 28394
Every cluster appears once: True


In [75]:
train_clusters = all_smpls_cluster_fixed[0][0][0][0]
test_clusters = all_smpls_cluster_fixed[0][0][1][0]

print("Training clusters:", len(train_clusters))
print("Testing clusters:", len(test_clusters))

print("Training unique:", len(np.unique(train_clusters)))
print("Testing unique:", len(np.unique(test_clusters)))

print("Training-test overlap:",
      len(np.intersect1d(train_clusters, test_clusters)))

print("Training + testing:",
      len(train_clusters) + len(test_clusters))

print("Total clusters:", len(unique_cluster_ids))

Training clusters: 22716
Testing clusters: 5678
Training unique: 22716
Testing unique: 5678
Training-test overlap: 0
Training + testing: 28394
Total clusters: 28394


In [76]:
for i in range(5):
    train_clusters = all_smpls_cluster_fixed[0][i][0][0]
    test_clusters = all_smpls_cluster_fixed[0][i][1][0]

    combined = np.concatenate(
        [train_clusters, test_clusters]
    )

    print(
        f"Fold {i+1}: "
        f"train={len(train_clusters)}, "
        f"test={len(test_clusters)}, "
        f"unique combined={len(np.unique(combined))}, "
        f"total combined={len(combined)}"
    )

Fold 1: train=22716, test=5678, unique combined=28394, total combined=28394
Fold 2: train=22715, test=5679, unique combined=28394, total combined=28394
Fold 3: train=22715, test=5679, unique combined=28394, total combined=28394
Fold 4: train=22715, test=5679, unique combined=28394, total combined=28394
Fold 5: train=22715, test=5679, unique combined=28394, total combined=28394


In [77]:
import inspect
import doubleml.utils._checks as checks

print(inspect.getsource(checks._check_cluster_sample_splitting))

def _check_cluster_sample_splitting(all_smpls_cluster, dml_data, n_rep, n_folds):
    if all_smpls_cluster is None:
        raise ValueError("For cluster data, all_smpls_cluster must be provided.")

    n_rep_cluster = len(all_smpls_cluster)
    if n_rep_cluster != n_rep:
        raise ValueError(
            "Invalid samples provided. Number of repetitions for all_smpls and all_smpls_cluster must be the same."
        )

    for i_rep in range(n_rep):
        n_folds_cluster = len(all_smpls_cluster[i_rep])
        if n_folds_cluster != n_folds:
            raise ValueError("Invalid samples provided. Number of folds for all_smpls and all_smpls_cluster must be the same.")
        for i_cluster in range(dml_data.n_cluster_vars):
            this_cluster_var = dml_data.cluster_vars[:, i_cluster]
            clusters = np.unique(this_cluster_var)
            cluster_partition = [all_smpls_cluster[0][0][0][i_cluster], all_smpls_cluster[0][0][1][i_cluster]]
            is_cluster_partition =

In [78]:
# Inspect the original cluster labels and our mapped indices
original_clusters = np.unique(dml_data.cluster_vars[:, 0])

print("Original cluster labels (first 5):")
print(original_clusters[:5])

print("\nMapped cluster indices (first 5):")
print(unique_cluster_ids[:5])

print("\nAre these arrays identical?")
print(np.array_equal(original_clusters, unique_cluster_ids))

Original cluster labels (first 5):
['100_1313000002' '100_1313000005' '100_1313000032' '100_1313000050'
 '100_1313000054']

Mapped cluster indices (first 5):
['100_1313000002' '100_1313000005' '100_1313000032' '100_1313000050'
 '100_1313000054']

Are these arrays identical?
True


In [80]:
first_fold = all_smpls_cluster_fixed[0][0]

print("Fold type:", type(first_fold))
print("Number of elements in fold:", len(first_fold))

print("\nTraining entry type:", type(first_fold[0]))
print("Testing entry type:", type(first_fold[1]))

print("\nTraining entry shape/type:")
print(type(first_fold[0][0]))
print("\nTesting entry shape/type:")
print(type(first_fold[1][0]))

Fold type: <class 'tuple'>
Number of elements in fold: 2

Training entry type: <class 'list'>
Testing entry type: <class 'list'>

Training entry shape/type:
<class 'numpy.ndarray'>

Testing entry shape/type:
<class 'numpy.ndarray'>


In [81]:
# Inspect the first fold's cluster arrays
train_clusters = np.asarray(first_fold[0][0])
test_clusters = np.asarray(first_fold[1][0])

expected_clusters = np.unique(dml_data.cluster_vars[:, 0])

print("Training cluster dtype:", train_clusters.dtype)
print("Testing cluster dtype:", test_clusters.dtype)
print("Expected cluster dtype:", expected_clusters.dtype)

print("\nFirst 5 training values:", train_clusters[:5])
print("First 5 testing values:", test_clusters[:5])
print("First 5 expected labels:", expected_clusters[:5])

print("\nTraining + testing count:",
      len(train_clusters) + len(test_clusters))
print("Expected unique clusters:", len(expected_clusters))

print("\nDoes the combined split match all expected labels?",
      np.array_equal(
          np.sort(np.concatenate([train_clusters, test_clusters])),
          np.sort(expected_clusters)
      ))

Training cluster dtype: int64
Testing cluster dtype: int64
Expected cluster dtype: object

First 5 training values: [0 1 2 3 4]
First 5 testing values: [12 13 16 17 21]
First 5 expected labels: ['100_1313000002' '100_1313000005' '100_1313000032' '100_1313000050'
 '100_1313000054']

Training + testing count: 28394
Expected unique clusters: 28394

Does the combined split match all expected labels? False


In [82]:
# Get the original cluster label for every row
row_cluster_labels = dml_data.cluster_vars[:, 0]

# Rebuild cluster splits using original labels
all_smpls_cluster_fixed = [[
    (
        [
            np.unique(row_cluster_labels[train_idx])
        ],
        [
            np.unique(row_cluster_labels[test_idx])
        ]
    )
    for train_idx, test_idx in folds
]]

# Check the first fold
train_clusters = all_smpls_cluster_fixed[0][0][0][0]
test_clusters = all_smpls_cluster_fixed[0][0][1][0]

print("Training cluster dtype:", train_clusters.dtype)
print("Testing cluster dtype:", test_clusters.dtype)
print("Training clusters:", len(train_clusters))
print("Testing clusters:", len(test_clusters))

combined = np.concatenate([train_clusters, test_clusters])
expected = np.unique(row_cluster_labels)

print("Matches all original labels:",
      np.array_equal(np.sort(combined), np.sort(expected)))

Training cluster dtype: object
Testing cluster dtype: object
Training clusters: 22716
Testing clusters: 5678
Matches all original labels: True


In [83]:
dml_model.set_sample_splitting(
    all_smpls,
    all_smpls_cluster=all_smpls_cluster_fixed
)

print("DoubleML accepted the cluster-aware folds!")
print("Number of folds:", dml_model.n_folds)
print("Number of repetitions:", dml_model.n_rep)

DoubleML accepted the cluster-aware folds!
Number of folds: 5
Number of repetitions: 1


In [84]:
print("Outcome pipeline (ml_l):")
print(ml_l)

print("\nTreatment pipeline (ml_m):")
print(ml_m)

print("\nAre both pipelines created?",
      ml_l is not None and ml_m is not None)

Outcome pipeline (ml_l):
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['STORE', 'UPC', 'WEEK'])])),
                ('model',
                 RandomForestRegressor(min_samples_leaf=10, n_jobs=-1,
                                       random_state=42))])

Treatment pipeline (ml_m):
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['STORE', 'UPC', 'WEEK'])])),
                ('model',
                 RandomForestRegressor(min_samples_leaf=10, n_jobs=-1,
                                       random_state=42))])

Are both pipelines created? True


In [86]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# All three controls are categorical
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            [0, 1, 2]
        )
    ]
)

# Outcome model: predict changes in movement
ml_l = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42
    ))
])

# Treatment model: predict changes in price
ml_m = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42
    ))
])

print("Both pipelines rebuilt using column positions!")

Both pipelines rebuilt using column positions!


In [87]:
# Create a fresh DoubleML model
dml_model = dml.DoubleMLPLR(
    dml_data,
    ml_l=ml_l,
    ml_m=ml_m,
    n_folds=5,
    score="partialling out"
)

# Reapply our validated cluster-aware folds
dml_model.set_sample_splitting(
    all_smpls,
    all_smpls_cluster=all_smpls_cluster_fixed
)

print("New DoubleML model created successfully!")
print("Folds:", dml_model.n_folds)
print("Repetitions:", dml_model.n_rep)

New DoubleML model created successfully!
Folds: 5
Repetitions: 1
